# PhiloVista-1800 Annotation Quality Audit

## tl;dr

All 1,800 item records now pass structural, mapping, image-integrity, English-only, separator, export-consistency, and sensitive-data checks. The audit corrected six confirmed cross-image annotation mismatches, 26 boundary-track overinterpretation gaps, and one excessive caption. Seventeen medium-severity review reminders remain; none is a confirmed factual or structural error. These records remain non-independent AI drafts rather than human gold annotations.

## Context & Methods

The unit of analysis is one annotation JSON per frozen blind image. The audit compares item records with the batch plan, blind image index, final-selection manifest, mapped image bytes, and three exported files.

### Key Assumptions

- `formal_gold=false` and both human-review flags must remain set.
- Scene, Action, and Rationale require exactly three English references; Object requires five.
- Boundary-track items need at least one interpretation that explicitly records insufficient evidence.
- Repeated but accurate negative observations and mirrored actions are review reminders, not automatic errors.

In [1]:
import csv
import json
from collections import Counter
from pathlib import Path

workspace = Path.cwd()
v3_root = workspace / "NewBenchmark" / "PhilosophyHL_v3"
workflow = v3_root / "annotation_workflow"
audit_dir = v3_root / "audit" / "annotation_quality_v1"
report = json.loads((audit_dir / "PhiloVista-1800_quality_report.json").read_text(encoding="utf-8"))
with (audit_dir / "PhiloVista-1800_remaining_issues.csv").open(encoding="utf-8-sig", newline="") as handle:
    issues = list(csv.DictReader(handle))
repairs = [json.loads(line) for line in (audit_dir / "PhiloVista-1800_repair_log.jsonl").read_text(encoding="utf-8").splitlines() if line]
print({
    "expected_items": report["expected_items"],
    "parsed_items": report["parsed_items"],
    "unique_ids": report["unique_annotation_item_ids"],
    "remaining_by_severity": report["issues"]["by_severity"],
    "unique_repaired_items": len({row["annotation_item_id"] for row in repairs}),
    "release_readiness": report["release_readiness"],
})

{'expected_items': 1800, 'parsed_items': 1800, 'unique_ids': 1800, 'remaining_by_severity': {'medium': 17}, 'unique_repaired_items': 33, 'release_readiness': 'conditionally_ready'}


## Data

The frozen population contains 600 HL Dataset images, 550 HAIVMet images, 500 IRFL images, and 150 MM-MoralBench images. It contains 1,620 positive-track and 180 boundary-track items, arranged as 36 batches of 50.

In [2]:
print("Source distribution:", report["source_distribution"])
print("Track distribution:", report["track_distribution"])
print("Method distribution:", report["method_distribution"])
print("Caption word-length profile:", report["caption_word_length"])

Source distribution: {'HAIVMet': 550, 'HL Dataset': 600, 'IRFL': 500, 'MM-MoralBench': 150}
Track distribution: {'boundary': 180, 'positive': 1620}
Method distribution: {'ai_agent_direct_visual_reannotation_after_audit_non_independent': 6, 'dual_api_deepseek_generate_glm_review_non_independent': 2, 'dual_api_deepseek_generate_glm_visual_review_non_independent': 1, 'dual_api_qwen_generate_deepseek_review_non_independent': 1, 'dual_api_qwen_generate_glm_review_non_independent': 1, 'dual_api_qwen_generate_glm_visual_review_non_independent': 1789}
Caption word-length profile: {'action': {'min': 6, 'median': 16.0, 'p90': 23, 'max': 74}, 'object': {'min': 8, 'median': 21.0, 'p90': 29, 'max': 79}, 'rationale': {'min': 9, 'median': 20.0, 'p90': 26, 'max': 58}, 'scene': {'min': 5, 'median': 17.0, 'p90': 24, 'max': 52}}


## Results

The remaining automated reminders are intentionally not auto-rewritten: 16 records use one of two accurate sentences stating that no readable text is present, and one record describes symmetric left/right actions with nearly identical token sets. Automatic rewriting would risk introducing hallucinations merely to increase surface diversity.

In [3]:
remaining = Counter(row["code"] for row in issues)
print("Remaining review reminders:", dict(sorted(remaining.items())))

export_dir = workflow / "drafts" / "api_en" / "exports"
with (export_dir / "PhiloVista-1800_HL.csv").open(encoding="utf-8-sig", newline="") as handle:
    csv_rows = list(csv.DictReader(handle))
hl_rows = [json.loads(line) for line in (export_dir / "PhiloVista-1800_HL.jsonl").read_text(encoding="utf-8").splitlines() if line]
philosophy_rows = [json.loads(line) for line in (export_dir / "PhiloVista-1800_philosophy.jsonl").read_text(encoding="utf-8").splitlines() if line]
assert len(csv_rows) == len(hl_rows) == len(philosophy_rows) == 1800
assert not report["issues"]["by_severity"].get("critical", 0)
assert not report["issues"]["by_severity"].get("high", 0)
print("Export row counts and critical/high severity assertions passed.")

Remaining review reminders: {'cross_image_exact_reuse': 16, 'within_axis_near_duplicate': 1}
Export row counts and critical/high severity assertions passed.


## Takeaways

- The seven confirmed content/length defects and 26 boundary-labeling defects were repaired with backups and SHA-256 revision receipts.
- The hardened exporter now validates the full dataset in memory and atomically replaces outputs only after all checks pass.
- The AI draft is technically consistent and conditionally ready for human review, but it is not eligible for formal HL release until independent human annotation/rating and the confidence, purity, and diversity procedures are completed.